In [16]:
import json


class Cluster:
    def __init__(self, representative, members):
        self.representative = representative
        self.members = members

    def __repr__(self):
        return (
            f"Cluster(representative='{self.representative}', members={self.members})"
        )


def clean_name(name):
    return name.split("/")[-1].split(".")[0]


def load_clusters(array):
    result = []
    for obj in array:
        representative = clean_name(obj["representative"])
        members = list(map(clean_name, obj["members"]))
        result.append(Cluster(representative, members))
    return result


def load_all_clusters():
    for mode in ["approximate"]:
        for method in [
            "hierarchical",
            "affinity-propagation",
            "facility-location",
        ]:
            path = f"{mode}-{method}.json"
            with open(path) as f:
                data = json.load(f)
                yield (mode, method, load_clusters(data["clustering"]["clusters"]))


clustering = {}
for mode, method, clusters in load_all_clusters():
    clustering[(mode, method)] = clusters

print(f"Loaded {len(clustering)} clustering variants")
for (mode, method), clusters in clustering.items():
    print(f"  {mode} {method}: {len(clusters)} clusters")


Loaded 3 clustering variants
  approximate hierarchical: 544 clusters
  approximate affinity-propagation: 25 clusters
  approximate facility-location: 256 clusters


In [17]:
import pandas as pd

df = pd.read_csv("geometric_features.csv")
print(f"Dataset shape: {df.shape}")
print(f"Positive (tetrad=True): {df['tetrad'].sum()}")
print(f"Negative (tetrad=False): {(~df['tetrad']).sum()}")
df.head()


Dataset shape: (8807, 23)
Positive (tetrad=True): 2541
Negative (tetrad=False): 6266


,d01,d02,d03,d12,d13,d23,a012,as012,aa012,a013,...,as023,aa023,a123,as123,aa123,t0123,ts0123,ta0123,source_file,tetrad
0,11.599962,16.380459,11.585704,11.569917,16.368533,11.558636,1.570414,1.000000,0.000382,0.786379,...,0.707287,0.706927,1.572531,0.999998,-0.001734,0.004258,0.004258,0.999991,G4_139d-assembly1_000,True
1,11.494443,16.259659,11.508277,11.495788,16.272030,11.508291,1.571179,1.000000,-0.000383,0.785590,...,0.707780,0.706433,1.571496,1.000000,-0.000699,-0.002311,-0.002311,0.999997,G4_139d-assembly1_001,True
2,11.563400,16.342267,11.536212,11.544840,16.337894,11.572383,1.571074,1.000000,-0.000278,0.783977,...,0.705913,0.708299,1.569761,0.999999,0.001035,-0.004857,-0.004857,0.999988,G4_139d-assembly1_002,True
3,11.491617,16.210478,11.457415,11.474521,16.235972,11.462712,1.567223,0.999994,0.003573,0.783383,...,0.706791,0.707423,1.572881,0.999998,-0.002085,-0.001928,-0.001928,0.999998,G4_139d-assembly1_003,True
4,11.430656,15.902359,11.277481,11.525128,16.265011,11.305790,1.530541,0.999190,0.040244,0.765813,...,0.709146,0.705062,1.585767,0.999888,-0.014970,0.086127,0.086020,0.996293,G4_143d-assembly1_000,True


In [18]:
import itertools

N = 4

# Generate column names to drop (raw angle values, because we have the sine and cosine of these angles)
columns_to_drop = []

# Drop planar angle columns a{i}{j}{k}
for i, j, k in itertools.combinations(range(N), 3):
    columns_to_drop.append(f"a{i}{j}{k}")

# Drop torsion angle columns t{i}{j}{k}{l}
for i, j, k, l in itertools.combinations(range(N), 4):
    columns_to_drop.append(f"t{i}{j}{k}{l}")

df_filtered = df.drop(columns=columns_to_drop)
print(f"Original columns: {len(df.columns)}")
print(f"Filtered columns: {len(df_filtered.columns)}")
print(
    f"Feature columns: {len(df_filtered.columns) - 2}  (excluding source_file and tetrad)"
)


Original columns: 23
Filtered columns: 18
Feature columns: 16  (excluding source_file and tetrad)


In [19]:
from sklearn.model_selection import train_test_split

positive = df_filtered[df_filtered["tetrad"]]
negative = df_filtered[~df_filtered["tetrad"]]
splits = {}

for (mode, method), clusters in clustering.items():
    clusters.sort(key=lambda c: len(c.members))

    positive_test_names = []

    for cluster in clusters:
        positive_test_names.extend([cluster.representative] + cluster.members)
        if len(positive_test_names) >= 0.25 * len(positive):
            break

    positive_train = positive[~positive["source_file"].isin(positive_test_names)]
    positive_test = positive[positive["source_file"].isin(positive_test_names)]

    negative_train, negative_test = train_test_split(
        negative, test_size=0.25, random_state=42
    )

    df_train = pd.concat([positive_train, negative_train])
    df_test = pd.concat([positive_test, negative_test])

    X_train = df_train.drop(columns=["source_file", "tetrad"])
    X_test = df_test.drop(columns=["source_file", "tetrad"])
    y_train = df_train["tetrad"]
    y_test = df_test["tetrad"]
    splits[(mode, method)] = (X_train, y_train, X_test, y_test)
    print(
        f"{mode} {method}: train={len(df_train)} (pos {len(positive_train)}, neg {len(negative_train)}), test={len(df_test)} (pos {len(positive_test)}, neg {len(negative_test)})"
    )


approximate hierarchical: train=6602 (pos 1903, neg 4699), test=2205 (pos 638, neg 1567)
approximate affinity-propagation: train=6577 (pos 1878, neg 4699), test=2230 (pos 663, neg 1567)
approximate facility-location: train=6599 (pos 1900, neg 4699), test=2208 (pos 641, neg 1567)


In [20]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

classifiers = {
    "Naive Bayes": GaussianNB(),
    "Logistic Regression": LogisticRegression(random_state=42, max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "SVM": SVC(random_state=42, probability=True),
}


def evaluate_classifier(X_train, y_train, X_test, y_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    for name, classifier in classifiers.items():
        classifier.fit(X_train_scaled, y_train)
        y_pred = classifier.predict(X_test_scaled)
        yield (
            name,
            classifier,
            scaler,
            classification_report(y_test, y_pred, output_dict=True),
        )


In [21]:
from pickle import dump

N = 4

for (mode, method), (X_train, y_train, X_test, y_test) in splits.items():
    print(f"\n=== Evaluating {mode} {method} ===")
    for name, clf, scaler, report in evaluate_classifier(
        X_train, y_train, X_test, y_test
    ):
        print(f"  Classifier: {name}")
        print(
            f"    accuracy={report['accuracy']:.4f}  "
            f"pos_precision={report['True']['precision']:.4f}  "
            f"pos_recall={report['True']['recall']:.4f}  "
            f"pos_f1={report['True']['f1-score']:.4f}"
        )

        model_name = name.lower().replace(" ", "-")
        payload = {
            "classifier_name": name,
            "classifier": clf,
            "scaler": scaler,
            "feature_columns": list(X_train.columns),
            "window_size": N,
            "positive_label": True,
            "split_mode": mode,
            "split_method": method,
        }

        with open(f"{mode}-{method}-{model_name}.pkl", "wb") as f:
            dump(payload, f)

print("\nDone. Trained and saved 5 classifiers x 3 splits = 15 model bundles.")



=== Evaluating approximate hierarchical ===
  Classifier: Naive Bayes
    accuracy=0.9270  pos_precision=0.9857  pos_recall=0.7586  pos_f1=0.8574
  Classifier: Logistic Regression
    accuracy=0.9701  pos_precision=0.9181  pos_recall=0.9843  pos_f1=0.9501
  Classifier: Decision Tree
    accuracy=0.9283  pos_precision=0.9781  pos_recall=0.7696  pos_f1=0.8614
  Classifier: Random Forest
    accuracy=0.9424  pos_precision=0.9942  pos_recall=0.8056  pos_f1=0.8900
  Classifier: SVM
    accuracy=0.9828  pos_precision=0.9587  pos_recall=0.9828  pos_f1=0.9706

=== Evaluating approximate affinity-propagation ===
  Classifier: Naive Bayes
    accuracy=0.9318  pos_precision=0.9830  pos_recall=0.7843  pos_f1=0.8725
  Classifier: Logistic Regression
    accuracy=0.9717  pos_precision=0.9225  pos_recall=0.9879  pos_f1=0.9541
  Classifier: Decision Tree
    accuracy=0.9475  pos_precision=0.9910  pos_recall=0.8311  pos_f1=0.9040
  Classifier: Random Forest
    accuracy=0.9439  pos_precision=0.9927  p